In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

class MacroForecaster:
    """
    Module 1: Dự báo kinh tế vĩ mô Việt Nam 2026-2030 (AIDEOM-VN).
    Sử dụng hàm sản xuất Cobb-Douglas mở rộng để dự báo GDP.
    """
    def __init__(self):
        # Các hệ số độ co giãn từ đề bài (Bài 1.3 và 10.3)
        self.alpha = 0.33  # Vốn vật chất (K)
        self.beta = 0.42   # Lao động (L)
        self.gamma = 0.10  # Mức độ số hóa (D)
        self.delta = 0.08  # Năng lực AI (AI)
        self.theta = 0.07  # Vốn nhân lực số (H)

    def load_data(self):
        """
        Nạp dữ liệu từ thư mục data/ nằm cùng cấp với thư mục notebooks/
        Sử dụng pathlib để xử lý đường dẫn linh hoạt.
        """
        # Lấy đường dẫn thư mục gốc của dự án (parent của thư mục notebooks hiện tại)
        base_dir = Path.cwd().parent
        data_path = base_dir / 'data' / 'vietnam_macro_2020_2025.csv'
        
        if not data_path.exists():
            raise FileNotFoundError(f"Không tìm thấy tệp dữ liệu tại: {data_path}")
            
        df = pd.read_csv(data_path)
        return df

    def calculate_tfp_series(self, df):
        """
        Tính toán năng suất nhân tố tổng hợp (A_t) cho giai đoạn 2020-2025.
        Công thức: A = Y / (K^alpha * L^beta * D^gamma * AI^delta * H^theta)
        """
        # Dữ liệu bổ sung không có trong CSV nhưng có trong yêu cầu Bài 1.3
        K = np.array([8044.4, 8487.5, 9513.3, 10221.8, 11511.9, 12847.6])
        L = np.array([53.6, 50.5, 51.7, 52.4, 52.9, 53.4])
        D = np.array([12.0, 12.7, 14.3, 16.5, 18.3, 19.5])
        AI = np.array([55.6, 60.2, 65.4, 67.0, 73.8, 80.1])
        H = np.array([24.1, 26.1, 26.2, 27.0, 28.4, 29.2])
        Y = df['GDP_trillion_VND'].values

        # Tính A_t cho từng năm [5]
        A_t = Y / (K**self.alpha * L**self.beta * D**self.gamma * AI**self.delta * H**self.theta)
        return A_t

    def forecast_2030(self, avg_A, last_values):
        """
        Mô phỏng và dự báo GDP Việt Nam năm 2030 theo kịch bản Bài 1.4.4.
        """
        # Các tham số kịch bản 2030 [6]
        years = np.arange(2026, 2031)
        k_growth, l_growth, tfp_growth = 0.06, 0.06, 0.012
        d_target, ai_target, h_target = 30.0, 100.0, 35.0
        
        # Giá trị năm gốc 2025
        K_t, L_t, A_t = 25900, 53.4, avg_A
        D_0, AI_0, H_0 = 19.5, 80.1, 29.2
        
        forecast_results = []
        
        for year in years:
            t = year - 2025
            # Tăng trưởng các yếu tố theo thời gian
            K_t *= (1 + k_growth)
            L_t *= (1 + l_growth)
            A_t *= (1 + tfp_growth)
            
            # Lộ trình tuyến tính đạt mục tiêu vào 2030
            D_t = D_0 + (d_target - D_0) * (t / 5)
            AI_t = AI_0 + (ai_target - AI_0) * (t / 5)
            H_t = H_0 + (h_target - H_0) * (t / 5)
            
            # Tính GDP dự báo [3]
            Y_hat = A_t * (K_t**self.alpha) * (L_t**self.beta) * \
                    (D_t**self.gamma) * (AI_t**self.delta) * (H_t**self.theta)
            
            # ---> SỬA Ở ĐÂY: Thêm TFP và Lao động vào Dictionary để xuất ra bảng <---
            forecast_results.append({
                'Năm': year,
                'GDP_Dự_báo': round(Y_hat, 2),
                'TFP_(A_t)': round(A_t, 4),
                'Lao_động_Triệu_người': round(L_t, 2)
            })
            
        return pd.DataFrame(forecast_results)

# --- Thực thi Module 1 ---
if __name__ == "__main__":
    try:
        forecaster = MacroForecaster()
        
        # 1. Nạp dữ liệu (Giữ nguyên cách tìm file như yêu cầu)
        df_history = forecaster.load_data()
        
        # 2. Tính TFP lịch sử
        A_series = forecaster.calculate_tfp_series(df_history)
        avg_A = np.mean(A_series)
        
        print("-" * 65)
        print(f"TFP trung bình giai đoạn 2020-2025: {avg_A:.4f}")
        
        # 3. Dự báo GDP, TFP, Lao động đến 2030
        df_2030 = forecaster.forecast_2030(avg_A, None)
        
        print("\nKẾT QUẢ ĐẦU RA MODULE 1 (GIAI ĐOẠN 2026-2030):")
        print(df_2030.to_string(index=False))
        print("-" * 65)
        
    except FileNotFoundError as e:
        print(e)
    except Exception as e:
        print(f"Đã xảy ra lỗi không xác định: {e}")

-----------------------------------------------------------------
TFP trung bình giai đoạn 2020-2025: 39.2582

KẾT QUẢ ĐẦU RA MODULE 1 (GIAI ĐOẠN 2026-2030):
 Năm  GDP_Dự_báo  TFP_(A_t)  Lao_động_Triệu_người
2026    15532.06    39.7293                 56.60
2027    16678.81    40.2061                 60.00
2028    17891.50    40.6886                 63.60
2029    19175.04    41.1768                 67.42
2030    20534.47    41.6710                 71.46
-----------------------------------------------------------------
